In [1]:
# @title 1. Hubungkan Google Drive
# @markdown Klik tombol "Play" di kiri, lalu ikuti instruksi pop-up untuk menghubungkan akun Google Drive Anda.

from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Google Drive berhasil terhubung!")

Mounted at /content/drive
✅ Google Drive berhasil terhubung!


In [ ]:
# @title 2. Danbooru Downloader
# @markdown Isi form di bawah ini, lalu tekan tombol Play (▶) di sebelah kiri.

import requests
import time
import os
from tqdm.notebook import tqdm

# --- FORM CONFIGURATION ---
search_tags = "hatsune_miku" # @param {type:"string"}
rating_filter = "Semua (Tanpa Filter)" # @param ["General (Aman)", "Sensitive (R-15)", "Questionable (R-18ish)", "Explicit (R-18)", "Semua (Tanpa Filter)"]
max_pages = 3 # @param {type:"slider", min:1, max:50, step:1}
limit_per_page = 20 # @param {type:"slider", min:10, max:100, step:10}
simpan_ke_folder_khusus = True # @param {type:"boolean"}

# --- LOGIC CODE ---

def get_rating_tag(selection):
    if selection == "General (Aman)": return " rating:general"
    if selection == "Sensitive (R-15)": return " rating:sensitive"
    if selection == "Questionable (R-18ish)": return " rating:questionable"
    if selection == "Explicit (R-18)": return " rating:explicit"
    return "" # Untuk "Semua"

def sanitize_filename(name):
    return "".join([c for c in name if c.isalpha() or c.isdigit() or c==' ' or c=='_']).strip()

def run_scraper():
    # 1. Siapkan Tag
    final_tags = search_tags + get_rating_tag(rating_filter)

    # 2. Siapkan Folder
    base_folder = "/content/drive/MyDrive/Danbooru_Download"

    if simpan_ke_folder_khusus:
        # Membuat sub-folder berdasarkan nama tag agar rapi
        folder_name = sanitize_filename(search_tags)
        save_path = os.path.join(base_folder, folder_name)
    else:
        save_path = base_folder

    if not os.path.exists(save_path):
        os.makedirs(save_path)

    print(f"📂 Menyimpan ke: {save_path}")
    print(f"🔍 Mencari tag: [{final_tags}]")
    print(f"📄 Halaman: 1 sampai {max_pages}")
    print("-" * 30)

    # 3. Setup Request
    base_url = "https://danbooru.donmai.us/posts.json"
    headers = {"User-Agent": "ColabUser/1.0"}

    total_downloaded = 0

    # 4. Loop Halaman
    for page in range(1, max_pages + 1):
        params = {
            "tags": final_tags,
            "limit": limit_per_page,
            "page": page
        }

        try:
            req = requests.get(base_url, params=params, headers=headers)
            if req.status_code != 200:
                print(f"❌ Error akses API (Page {page}): {req.status_code}")
                break

            posts = req.json()
            if not posts:
                print("⚠️ Tidak ada gambar lagi. Berhenti.")
                break

            # Progress Bar per halaman
            for post in tqdm(posts, desc=f"Halaman {page}", leave=False):
                if "file_url" in post:
                    file_url = post["file_url"]
                    file_ext = post["file_ext"]
                    post_id = post["id"]

                    filename = os.path.join(save_path, f"{post_id}.{file_ext}")

                    if not os.path.exists(filename):
                        # Download
                        img_data = requests.get(file_url, headers=headers).content
                        with open(filename, 'wb') as f:
                            f.write(img_data)
                        total_downloaded += 1
                        time.sleep(0.5) # Jeda sopan

            time.sleep(1) # Jeda antar halaman

        except Exception as e:
            print(f"Error: {e}")

    print("-" * 30)
    print(f"✅ Selesai! Total {total_downloaded} gambar baru tersimpan.")

if __name__ == "__main__":
    run_scraper()

In [6]:
# @title 4. Danbooru Multi-Tag Downloader
# @markdown **Cara isi tag:** Pisahkan dengan SPASI. <br>
# @markdown Contoh: `hatsune_miku swimsuit` (Max 2 tag agar aman).

import requests
import time
import os
from tqdm.notebook import tqdm

# --- 1. KONFIGURASI FORM ---
search_tags = "lucia:_lotus_(pgr) solo" # @param {type:"string"}
nama_folder_tujuan = "Loras/LuciaLotus/dataset" # @param {type:"string"}
# @markdown Biarkan `nama_folder_tujuan` kosong jika ingin nama folder otomatis dari tag.

rating_filter = "Semua (Tanpa Filter)" # @param ["General (Aman)", "Sensitive (R-15)", "Questionable (R-18ish)", "Explicit (R-18)", "Semua (Tanpa Filter)"]
max_pages = 20 # @param {type:"slider", min:1, max:50, step:1}
limit_per_page = 50 # @param {type:"slider", min:10, max:100, step:10}

# --- 2. LOGIKA UTAMA ---

def get_rating_tag(selection):
    if selection == "General (Aman)": return " rating:general"
    if selection == "Sensitive (R-15)": return " rating:sensitive"
    if selection == "Questionable (R-18ish)": return " rating:questionable"
    if selection == "Explicit (R-18)": return " rating:explicit"
    return ""

def sanitize_folder_name(name):
    # Mengganti spasi dengan underscore agar nama folder rapi
    # Contoh: "hatsune_miku swimsuit" -> "hatsune_miku_swimsuit"
    clean_name = "".join([c for c in name if c.isalpha() or c.isdigit() or c in (' ', '_', '-')]).strip()
    return clean_name.replace(" ", "_")

def run_scraper():
    # A. Tentukan Nama Folder
    base_drive_path = "/content/drive/MyDrive"

    if nama_folder_tujuan.strip() != "":
        target_folder = sanitize_folder_name(nama_folder_tujuan)
    else:
        # Jika user tidak mengisi nama folder, kita buat dari tag
        # Kita ambil tag utama saja (search_tags) tanpa tag rating
        target_folder = sanitize_folder_name(search_tags)

    full_save_path = os.path.join(base_drive_path, nama_folder_tujuan)

    if not os.path.exists(full_save_path):
        os.makedirs(full_save_path)
        print(f"✅ Folder dibuat: {full_save_path}")
    else:
        print(f"📂 Masuk ke folder: {full_save_path}")

    # B. Susun Query API
    # Gabungkan tag pencarian user + tag rating
    final_query = search_tags + get_rating_tag(rating_filter)

    print(f"🔍 Query API: [{final_query}]")
    print("-" * 40)

    base_url = "https://danbooru.donmai.us/posts.json"
    headers = {"User-Agent": "ColabMultiTag/1.0"}
    total_downloaded = 0

    # C. Eksekusi Download
    for page in range(1, max_pages + 1):
        params = {
            "tags": final_query,
            "limit": limit_per_page,
            "page": page
        }

        try:
            req = requests.get(base_url, params=params, headers=headers)

            # Handling jika user memasukkan lebih dari 2 tag dan ditolak server
            if req.status_code == 422:
                print(f"❌ Error 422: Kemungkinan Anda memasukkan terlalu banyak tag (Maksimal 2 tag).")
                break
            if req.status_code != 200:
                print(f"❌ Gagal akses Hal {page} (Status: {req.status_code})")
                break

            posts = req.json()
            if not posts:
                print("⚠️ Tidak ada hasil. Cek ejaan tag atau kurangi jumlah tag.")
                break

            for post in tqdm(posts, desc=f"Halaman {page}", leave=False):
                if "file_url" in post:
                    file_url = post["file_url"]
                    file_ext = post["file_ext"]
                    post_id = post["id"]

                    filename = os.path.join(full_save_path, f"{post_id}.{file_ext}")

                    if not os.path.exists(filename):
                        try:
                            content = requests.get(file_url, headers=headers, timeout=10).content
                            with open(filename, 'wb') as f:
                                f.write(content)
                            total_downloaded += 1
                            time.sleep(0.5)
                        except Exception as e:
                            pass # Skip jika gagal download gambar spesifik

            time.sleep(1)

        except Exception as e:
            print(f"Error Script: {e}")

    print("-" * 40)
    print(f"🎉 Selesai! {total_downloaded} gambar tersimpan.")

if __name__ == "__main__":
    run_scraper()

📂 Masuk ke folder: /content/drive/MyDrive/Loras/LuciaLotus/dataset
🔍 Query API: [lucia:_lotus_(pgr) solo]
----------------------------------------


Halaman 1:   0%|          | 0/50 [00:00<?, ?it/s]

Halaman 2:   0%|          | 0/8 [00:00<?, ?it/s]

⚠️ Tidak ada hasil. Cek ejaan tag atau kurangi jumlah tag.
----------------------------------------
🎉 Selesai! 58 gambar tersimpan.
